# Đào tạo mạng SOTA CTR-GCN cho Hệ Thống Báo Động (Transfer Learning)

Notebook này được thiết kế để chạy tự động quy trình chuẩn hóa. Tất cả code logic đã được tách ra thành file `.py` chuyên nghiệp.

In [1]:
# 1. KẾT NỐI VỚI GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 2. TẢI MÃ NGUỒN TỪ GITHUB VÀ CÀI ĐẶT THƯ VIỆN
!if [ -d "realtime-abnormal-behavior-detection" ]; then cd realtime-abnormal-behavior-detection && git fetch origin && git reset --hard origin/dev; else git clone -b dev https://github.com/NBasLongz/realtime-abnormal-behavior-detection.git; fi
%cd /content/realtime-abnormal-behavior-detection
!pip install -r requirements.txt
!pip install onnx onnxscript onnxruntime

Cloning into 'realtime-abnormal-behavior-detection'...
remote: Enumerating objects: 258, done.
remote: Counting objects: 100% (258/258), done.
remote: Compressing objects: 100% (183/183), done.
remote: Total 258 (delta 104), reused 217 (delta 64), pack-reused 0 (from 0)
Receiving objects: 100% (258/258), 650.85 KiB | 14.79 MiB/s, done.
Resolving deltas: 100% (104/104), done.
/content/realtime-abnormal-behavior-detection
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 122.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 88.2 MB/s eta 0:00:00


In [3]:
# 3. CHÉP DỮ LIỆU TỪ DRIVE VÀO PROJECT
!cp -r /content/drive/MyDrive/Abnormal_Data/raw/* data/raw/

In [4]:
# 4. TRÍCH XUẤT KHUNG XƯƠNG (Pose Estimation)
!python scripts/data_processing/build_sequences.py

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
BUILDING SKELETON SEQUENCES (EXCLUDING LOITERING)

[NORMAL]
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 2 packages in 288ms
Prepared 1 package in 39ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

  Processed: normal_001.mp4
  Processed: normal_002.mp4
  Processed: normal_003.mp4
  Processed: normal_004.mp4
  Processed: normal_005.mp4
  Processed: normal_006.mp4
  Processed: normal_007.mp4
  Processed: normal_008.mp4
  Processed: normal_009.mp4
  Processed: normal_010.mp4
  Processe

In [5]:
# 5. CHUẨN HÓA TỌA ĐỘ VỀ TÂM CƠ THỂ [-1, 1], AUGMENTATION VÀ ĐỔI FORMAT CTR-GCN
!python scripts/data_processing/export_gcn_dataset.py

⏳ Đang đọc dữ liệu từ: /content/realtime-abnormal-behavior-detection/data/processed/sequences
⏳ Đang xử lý và chuyển đổi dữ liệu sang CTR-GCN Format...
Đã chuẩn bị dữ liệu GCN: X=(5748, 3, 30, 17), Y=(5748,)
Dữ liệu gốc: 1437 sequences
Sau khi Augment (x4): 5748 sequences
Hoàn thành! File được lưu tại: /content/realtime-abnormal-behavior-detection/data/processed/gcn


In [7]:
# 6. HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH (GỒM BÁO CÁO F1, RECALL, CONFUSION MATRIX)
# Lưu output vào cả console và file text để bạn làm báo cáo
!python scripts/training/train_ctr_gcn.py | tee training_report.txt

/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Bắt đầu quá trình Transfer Learning cho CTR-GCN...
Thiết bị huấn luyện: cuda
Dữ liệu đã tải: X=(5748, 3, 30, 17), Y=(5748,)
CẢNH BÁO: Không tìm thấy pretrained weights tại /content/realtime-abnormal-behavior-detection/weights/classification/ctrgcn_ntu_120.pt. Sẽ train từ đầu (Từ chối Transfer Learning).
Epoch 1/40 | Loss: 0.9347 | Val Accuracy: 0.5252 | Val Macro-F1: 0.5123
Đã lưu mô hình tốt nhất đạt Macro-F1: 0.5123
Epoch 2/40 | Loss: 0.8376 | Val Accuracy: 0.6009 | Val Macro-F1: 0.6138
Đã lưu mô hình tốt nhất đạt Macro-F1: 0.6138
Epoch 3/40 | Loss: 0.7846 | Val Accuracy: 0.6061 | Val Macro-F1: 0.5405
Epoch 4/40 | Loss: 0.7528 | Val Accuracy: 0.6383 | Val Macro-F1: 0.6551
Đã lưu mô hình tốt nhất đạt Macro-F1: 0.6551
Epoch 5/40 | Loss: 0.7250 | Val Accuracy: 0.6530 | Val Macro-F1: 0.6661


In [8]:
# 7. XUẤT ONNX VÀ SAO CHÉP MỌI KẾT QUẢ VỀ GOOGLE DRIVE
!python scripts/training/export_onnx.py

# a. Chép Model Weights (.onnx và .pt)
!cp weights/classification/ctrgcn_best.onnx /content/drive/MyDrive/Abnormal_Data/ 2>/dev/null || true
!cp weights/classification/ctrgcn_best.pt /content/drive/MyDrive/Abnormal_Data/ 2>/dev/null || true

# b. Chép Báo cáo đánh giá
!cp training_report.txt /content/drive/MyDrive/Abnormal_Data/ 2>/dev/null || true

# c. Nén toàn bộ dữ liệu khung xương đã chuẩn hóa
!zip -r processed_data.zip data/processed/
!cp processed_data.zip /content/drive/MyDrive/Abnormal_Data/ 2>/dev/null || true

print("TẤT CẢ KẾT QUẢ ĐÃ ĐƯỢC LƯU VỀ GOOGLE DRIVE CỦA BẠN!")

⏳ Đang xuất mô hình ra ONNX...
[torch.onnx] Obtain model graph for `CTRGCN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `CTRGCN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Xuất ONNX thành công tại: /content/realtime-abnormal-behavior-detection/weights/classification/ctrgcn_best.onnx
  adding: data/processed/ (stored 0%)
  adding: data/processed/sequences/ (stored 0%)
  adding: data/processed/sequences/normal/ (stored 0%)
  adding: data/processed/sequences/normal/normal_068_id571.npy (de